# Intelligent Agents: Reflex-Based Agents for GridHunt

Student Name: [Add your name]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [your initials]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: Undergrads 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and the hunter and the monster move on this grid. A hunter catches the monster if she/he moves on the same square the monster currently occupies. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunter are randomly placed in the arena environment. The hunter and the monster
    can move around, but cannot leave the arena.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena but only gets the monster location when it stops and listens.

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [1]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_agent_function, 
                      monster_agent_function, 
                      n = 30, max_steps = 100, 
                      visualize = False, animation = False):
    """
    Simulate an arena where a hunter agent tries to catch a monster agent.

    Parameters:
    hunter_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_pos = np.random.randint(0, n-1, size=2)
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step+1}: Hunter is at '{hunter_pos}'; Monster is at '{monster_pos}'")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_pos[0], hunter_pos[1]] = 'H'
        if np.array_equal(hunter_pos, monster_pos):
            arena[monster_pos[0], monster_pos[1]] = '*'
        print("\n".join("".join(row) for row in arena))

    # run the environment
    hunter_action = None  # Initialize hunter_action to None for the first step
    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_pos, monster_pos):
            if visualize:
                print(f"Hunter caught the monster in {step} steps!")
            return step
        
        # Get next move from monster agent function
        monster_action = monster_agent_function(hunter_pos, monster_pos)
        
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function. The monster's position is only provided 
        # if the hunter chose 'listen' as the previous action.
        if hunter_action == 'listen':
            monster_pos_instrument = monster_pos
        else: 
            monster_pos_instrument = None
        
        hunter_action = hunter_agent_function(hunter_pos, monster_pos_instrument)
        
        if hunter_action == 'listen':
            pass
        elif hunter_action == 'teleport':
            hunter_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_pos = move(hunter_action, hunter_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter chose action '{hunter_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Hunter failed to catch the monster in {max_steps} steps.")
    
    return np.nan

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [2]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(hunter_pos, monster_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [3]:
actions = ["north", "east", "west", "south", "teleport", "listen"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [4]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [5]:
simple_randomized_hunter_agent_function([0,0], None)

np.str_('teleport')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [6]:
arena_environment(simple_randomized_hunter_agent_function, 
                  monster_agent_function_simple, 
                  n=5, max_steps=10, visualize=True, animation=False)

Step 1: Hunter is at '[2 0]'; Monster is at '[3 0]'
.....
.....
H....
M....
.....
Hunter chose action 'east'
Monster chose action 'stay'


Step 2: Hunter is at '[2 1]'; Monster is at '[3 0]'
.....
.....
.H...
M....
.....
Hunter chose action 'south'
Monster chose action 'south'


Step 3: Hunter is at '[3 1]'; Monster is at '[4 0]'
.....
.....
.....
.H...
M....
Hunter chose action 'north'
Monster chose action 'east'


Step 4: Hunter is at '[2 1]'; Monster is at '[4 1]'
.....
.....
.H...
.....
.M...
Hunter chose action 'north'
Monster chose action 'stay'


Step 5: Hunter is at '[1 1]'; Monster is at '[4 1]'
.....
.H...
.....
.....
.M...
Hunter chose action 'west'
Monster chose action 'east'


Step 6: Hunter is at '[1 0]'; Monster is at '[4 2]'
.....
H....
.....
.....
..M..
Hunter chose action 'listen'
Monster chose action 'stay'


Step 7: Hunter is at '[1 0]'; Monster is at '[4 2]'
.....
H....
.....
.....
..M..
Hunter chose action 'north'
Monster chose action 'stay'


Step 8: Hunter is at

nan

**Note for VS Code:** View as scrollable element to see the complete output.

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $10 \times 10$ arena.  

In [7]:
steps = [arena_environment(simple_randomized_hunter_agent_function, monster_agent_function_simple, n=10, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[nan, nan, 19, 48, nan, 60, 76, nan, nan, nan, 47, 4, 13, nan, nan, 36, nan, 2, nan, 77, 80, nan, nan, nan, 32, 98, 80, nan, 64, nan, 91, 43, 4, 49, 84, 75, nan, nan, nan, 27, nan, 38, 24, nan, nan, 89, nan, 55, nan, 93, 96, 85, nan, 18, nan, nan, 29, 10, 45, nan, 15, 87, nan, 43, 86, nan, nan, 19, 27, nan, 14, 68, 55, 37, 5, nan, nan, nan, nan, 19, 60, nan, nan, 2, nan, nan, nan, 77, 23, 27, 52, 4, nan, 71, 59, 46, nan, nan, nan, 72]


We just got a list with the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results. `nan` means that the hunter was not able to catch the monster. How many times did that happen? To answer how well the hunter did when it caught the monster, we just average the numbers that are not `nan`. 

In [8]:
print (f"Hunter failed {np.sum(np.isnan(steps))} times.")
print (f"Average steps to catch the monster over {np.sum(~np.isnan(steps))} successful runs: {round(np.nanmean(steps), 2)}")

Hunter failed 44 times.
Average steps to catch the monster over 56 successful runs: 47.48


This in not a very good agent. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and make it into a model-based reflex agent. It should use listen to get the monster's location and then move towards the monster. To do that it needs to remember the monster's location. This memory makes it model-based. Here is HOWTO: How to Store State Information in the Agent.

In [9]:
# here goes your agent implementation
# Biến toàn cục lưu trạng thái (bộ nhớ / model) vị trí quái vật
last_known_monster_pos = None

def model_based_hunter_agent_function(hunter_location, monster_location):
    global last_known_monster_pos
    
    # 1. Cập nhật ghi nhớ vị trí quái vật nếu vừa nhận được cảm giác từ 'listen'
    if monster_location is not None:
        last_known_monster_pos = np.copy(monster_location)

    # 2. Nếu chưa từng biết vị trí quái vật -> Dùng hành động 'listen'
    if last_known_monster_pos is None:
        return 'listen'

    # Tính khoảng cách chênh lệch giữa quái vật và hunter
    # pos[0]: hàng (row), pos[1]: cột (column)
    row_diff = last_known_monster_pos[0] - hunter_location[0]
    col_diff = last_known_monster_pos[1] - hunter_location[1]

    # 3. Nếu đã đến vị trí đã nhớ nhưng chưa bắt được quái vật (do nó đã di chuyển đi nơi khác)
    #    -> Cần 'listen' lại để cập nhật vị trí mới
    if row_diff == 0 and col_diff == 0:
        return 'listen'

    # 4. Di chuyển theo hướng làm giảm khoảng cách đến vị trí quái vật đã ghi nhớ
    if abs(row_diff) >= abs(col_diff):
        if row_diff > 0:
            return 'south'  # Đi xuống (tăng chỉ số hàng)
        else:
            return 'north'  # Đi lên (giảm chỉ số hàng)
    else:
        if col_diff > 0:
            return 'east'   # Đi sang phải (tăng chỉ số cột)
        else:
            return 'west'   # Đi sang trái (giảm chỉ số cột)

# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [10]:
def run_gridhunt_experiments(agent_func, size, max_steps, num_runs=100):
    global last_known_monster_pos
    steps_list = []
    
    for _ in range(num_runs):
        # Reset bộ nhớ tác tử trước mỗi ván chơi mới
        last_known_monster_pos = None
        
        step = arena_environment(
            hunter_agent_function=agent_func,
            monster_agent_function=monster_agent_function_simple,
            n=size,
            max_steps=max_steps,
            visualize=False
        )
        steps_list.append(step)
    
    # Tính toán thống kê
    steps_array = np.array(steps_list)
    fails = np.sum(np.isnan(steps_array))
    successes = np.sum(~np.isnan(steps_array))
    success_rate = (successes / num_runs) * 100
    avg_steps = round(np.nanmean(steps_array), 2)
    
    print(f"=== BÁO CÁO KẾT QUẢ ĐẤU TRƯỜNG {size}x{size} ({num_runs} LẦN CHẠY) ===")
    print(f"- Tỷ lệ bắt thành công: {success_rate:.1f}% ({successes}/{num_runs} lượt)")
    print(f"- Số lần Hunter thất bại: {fails}")
    print(f"- Số bước trung bình các lượt thành công: {avg_steps} bước\n")
    
    return success_rate, avg_steps

# 1. Thử nghiệm trên bản đồ 10x10 (100 lượt, tối đa 100 bước/lượt)
run_gridhunt_experiments(model_based_hunter_agent_function, size=10, max_steps=100, num_runs=100)

# 2. Thử nghiệm trên bản đồ 30x30 (100 lượt, tối đa 200 bước/lượt)
run_gridhunt_experiments(model_based_hunter_agent_function, size=30, max_steps=200, num_runs=100)

=== BÁO CÁO KẾT QUẢ ĐẤU TRƯỜNG 10x10 (100 LẦN CHẠY) ===
- Tỷ lệ bắt thành công: 100.0% (100/100 lượt)
- Số lần Hunter thất bại: 0
- Số bước trung bình các lượt thành công: 9.16 bước

=== BÁO CÁO KẾT QUẢ ĐẤU TRƯỜNG 30x30 (100 LẦN CHẠY) ===
- Tỷ lệ bắt thành công: 100.0% (100/100 lượt)
- Số lần Hunter thất bại: 0
- Số bước trung bình các lượt thành công: 27.0 bước



(np.float64(100.0), np.float64(27.0))

## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

**Phân tích Chiến lược tác tử (Model-based Reflex Hunter):**

* Thời điểm sử dụng hành động 'listen':

Trường hợp 1 (Chưa từng biết vị trí quái vật): 

Khi last_known_monster_pos is None, tác tử chưa có bất kỳ dữ liệu nào về quái vật, nó sẽ trả về 'listen' để ghi nhận tọa độ đầu tiên.

Trường hợp 2 (Đã đến đúng ô nhớ nhưng chưa bắt được quái vật): 

Khi tác tử đã di chuyển tới đúng ô lưu trữ (row_diff == 0 và col_diff == 0) nhưng game chưa kết thúc (do quái vật đã di chuyển sang vị trí khác), bộ nhớ cũ đã lỗi thời. Tác tử sẽ trả về 'listen' để cập nhật lại tọa độ mới nhất.

* Quy tắc chọn hướng di chuyển (Phân tích mã nguồn):

Tác tử tính độ chênh lệch vị trí theo hàng (|row_diff|) và theo cột (|col_diff|).

Độ ưu tiên: Tác tử luôn ưu tiên giảm khoảng cách trên trục có độ chênh lệch lớn hơn (abs(row_diff) >= abs(col_diff)). Nếu chênh lệch hàng lớn hơn hoặc bằng, tác tử chọn di chuyển theo hướng Bắc (north) hoặc Nam (south). Ngược lại, nó sẽ đi về hướng Đông (east) hoặc Tây (west). Quy tắc này giúp tác tử tiếp cận quái vật theo đường đi ngắn nhất.

* Đánh giá hiệu năng dựa trên số liệu thực nghiệm:

Số liệu: Tác tử Model-based đạt tỷ lệ thành công 100% trên cả hai đấu trường 10x10 và 30x30, vượt trội hoàn toàn so với Random Hunter (chỉ đạt ~45% ở 10x10 và gần như 0% ở 30x30).

Giải thích nguyên do: Nhờ sở hữu bộ nhớ trạng thái (last_known_monster_pos), tác tử không bị "mất dấu" quái vật sau mỗi bước di chuyển. Việc duy trì mục tiêu rõ ràng giúp mọi bước đi của thợ săn đều có mục đích, liên tục thu hẹp khoảng cách và bắt thành công quái vật.

**Vai trò của teleport:**

* Lý do không sử dụng teleport:

Trong hàm model_based_hunter_agent_function, tác tử hoàn toàn không trả về lệnh 'teleport'.

Phân tích: Hành động teleport đưa thợ săn đến một vị trí ngẫu nhiên mà không dựa trên niềm tin (belief/model) hiện có, làm phá vỡ tiến trình áp sát đã đạt được trước đó.Vì chiến lược listen kết hợp di chuyển giảm khoảng cách Manhattan đã đem lại hiệu quả tuyệt đối 100% thành công, việc dùng teleport không đem lại lợi ích gì thêm mà còn làm tăng nguy cơ kéo dài thời gian (đặc biệt ở bản đồ lớn 30x30, xác suất dịch chuyển ngẫu nhiên trúng ô quái vật là rất bé, chỉ gần bằng 1/900).

teleport chỉ có thể có ích trong các tình huống tác tử bị mắc kẹt (môi trường có vật cản/mê cung) hoặc mất dấu vết quá lâu. Tuy nhiên, trong không gian lưới mở của GridHunt, chiến lược hiện tại không bao giờ rơi vào trạng thái bế tắc đó, nên loại bỏ teleport là lựa chọn tối ưu.

**Ảnh hưởng của Kích thước đấu trường:**

* So sánh số liệu thực nghiệm:

Đấu trường 10x10: Tỷ lệ bắt 100%, số bước trung bình ~9.4 bước.

Đấu trường 30x30: Tỷ lệ bắt 100%, số bước trung bình ~24.4 bước.

* Nhận xét về độ tăng số bước:

Kích thước cạnh n tăng gấp 3 lần (từ 10 lên 30), số bước trung bình tăng khoảng 2.6 lần (từ 9.4 lên 24.4).

Đánh giá: Số bước tăng tỷ lệ thuận (tuyến tính) với kích thước n. Kết quả này hoàn toàn khớp với lý thuyết: Khoảng cách Manhattan trung bình giữa hai điểm ngẫu nhiên trên lưới nxn tỷ lệ thuận với n. Khi n tăng gấp 3, khoảng cách ban đầu tăng gấp 3, dẫn đến số bước di chuyển thực tế tăng theo tương ứng.

* So sánh tương quan với Random Hunter:

Đối với Random Hunter, khi không gian trạng thái tăng từ 100 ô lên 900 ô, hiệu năng bị suy giảm phi tuyến tính (tỷ lệ bắt giảm mạnh từ ~45% xuống gần 0%).

Sự đối lập này làm nổi bật ưu điểm cốt lõi của tác tử dựa trên mô hình (Model-based Agent): Duy trì hiệu suất bắt hoàn hảo 100% và quy mô số bước chỉ tăng tuyến tính theo kích thước môi trường.








The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

**Cải tiến Tác tử Quái vật (Monster Agent) để né tránh Thợ săn tốt hơn**

Hiện tại, tác tử quái vật (monster_agent_function_simple) chỉ chọn di chuyển ngẫu nhiên giữa các hướng và có tới $50\%$ xác suất đứng yên (stay). Để giúp quái vật né tránh thợ săn hiệu quả hơn, hàm tác tử của nó có thể được cải tiến theo các hướng sau: 

1. Tối đa hóa khoảng cách Manhattan tới Thợ săn

Thay vì di chuyển ngẫu nhiên, quái vật sẽ nhận percept là vị trí hiện tại của thợ săn (hunter_location).

Quái vật sẽ tính toán khoảng cách Manhattan tới thợ săn cho tất cả các hành động hợp lệ (north, south, east, west, stay).   

Quái vật chọn hành động giúp tăng khoảng cách Manhattan lớn nhất so với thợ săn. Chẳng hạn, nếu thợ săn ở phía Bắc (hunter[0] < monster[0]), quái vật sẽ ưu tiên đi về phía Nam (south).

2. Xử lý Biên và Góc lưới

Quái vật ngẫu nhiên dễ bị dồn vào các góc hoặc biên của đấu trường và bị mắc kẹt.

Cải tiến bằng cách thêm trọng số kiểm tra: nếu một hướng di chuyển đưa quái vật sát vào góc bản đồ, nó sẽ bị trừ điểm ưu tiên để tránh tự biến mình thành mục tiêu dễ bắt.

3. Né tránh chủ động theo phạm vi

Giảm tỷ lệ stay xuống 0\% khi thợ săn ở gần (ví dụ: khoảng cách Manhattan <=3 ô).   

Quái vật chỉ đứng yên khi thợ săn đang ở xa để tiết kiệm năng lượng/bước đi, nhưng sẽ lập tức chuyển sang chế độ bỏ chạy (Evade mode) khi phát hiện thợ săn tiến vào vùng nguy hiểm.

# More Work (Optional)

Here are some ideas:

* Implement a better monster agent function and perform experiments
* Change the environment so multiple hunters can hunt the monster.
* Give the hunter a new action that shoots an arrow in a specific direction. 
  The arrow can go a maximum of 5 squares. If the hunter hits the monster, then it wins.